# CBC密码分组链接模式原理与实现

**摘要：** 详解CBC分组密码模式的原理、特性、安全优势及Python实现

- **类别：** 现代密码学
- **难度：** 中等
- **预计用时：** 2小时
- **先修：** ECB模式基础、分组密码概念、异或运算、PKCS#7填充
- **学习目标：** 掌握CBC模式核心原理

> 说明：本课程强调“可运行 + 可解释 + 可练习”。代码尽量仅使用 Python 标准库（Pyodide 友好）。

## 你将获得什么

- **链式加密 Chaining：** 分组间通过异或形成链接
- **需要IV Require IV：** 依赖随机初始化向量
- **语义安全 Secure：** 相同明文产生不同密文

## 学习路线图（从直觉到实现）

我们把学习过程拆成 4 层，每一层都尽量给出“可验证”的产物：

1. **直觉层**：能说清楚它解决什么安全目标，以及为什么需要“密钥/参数/随机性”。
2. **符号层**：能把关键变换写成一个简短公式，例如 $y=f(x,k)$ 或 $$y=f(x,k)\bmod n$$。
3. **实现层**：能写出可运行的函数/类，并通过至少 3 条 `assert` 自测。
4. **对抗层**：能指出它可能被怎么攻（至少一个思路），并用代码做一个最小实验验证。

> 你会发现：能通过“对抗层”的小实验，往往才算真正理解。


## 应用场景与安全性讨论（扩充阅读）

这一主题通常会在以下场景出现（不同主题侧重点不同）：

- **教学/入门**：用简化模型理解“密钥 + 变换”的思想。
- **工程系统**：用成熟算法与标准协议实现保密性/完整性/认证。
- **攻防分析**：通过已知攻击理解“为什么某些方案不再推荐”。

### 安全目标（Security Goals）

常见目标包括：

- **保密性（Confidentiality）**：未授权者无法获得明文信息。
- **完整性（Integrity）**：消息在传输中被篡改能被发现。
- **认证（Authentication）**：确认对方是谁（或确认数据来自谁）。
- **不可否认（Non-repudiation）**：事后不能否认曾经生成/发送过某信息（通常与签名相关）。

### 威胁模型（Threat Model）

做任何“安全性结论”之前，先明确攻击者能做什么：

- 只能看到密文？还是还能选择明文（chosen-plaintext）？
- 能否篡改、重放、插入消息？
- 能否获取部分密钥/随机数？是否存在侧信道？

> 调试提示：如果你觉得“算法没问题但结果不对”，先检查你实现的威胁模型是否一致——例如你是否在复用 nonce/IV，或者把编码/填充当成了算法的一部分。


## 环境准备

In [ ]:
# 课程通用导入（尽量只用标准库）
import math
import random
import string
import secrets
import hashlib
import hmac
import itertools
import statistics
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional, Iterable

random.seed(0)  # 为了让演示更可复现


## 热身：数据与表示

很多实现问题来自“数据表示”而不是算法本身。请确保你能区分：

- 文本：`str`
- 字节：`bytes`
- 整数：`int`

并能相互转换。

In [ ]:
msg = "Hello, 密码学"
b = msg.encode("utf-8")
print(type(msg), type(b))  # 预期输出：<class 'str'> <class 'bytes'>
print(b[:10])              # 预期输出：前10个字节（与编码有关）
print(b.hex()[:20])        # 预期输出：十六进制字符串前20个字符


### 工具函数：字节/整数/十六进制

In [ ]:
def bytes_to_hex(b: bytes) -> str:
    return b.hex()

def hex_to_bytes(h: str) -> bytes:
    h = h.strip().lower()
    if h.startswith("0x"):
        h = h[2:]
    return bytes.fromhex(h)

def int_to_bytes(x: int, length: int, byteorder: str = "big") -> bytes:
    return x.to_bytes(length, byteorder=byteorder, signed=False)

def bytes_to_int(b: bytes, byteorder: str = "big") -> int:
    return int.from_bytes(b, byteorder=byteorder, signed=False)

assert hex_to_bytes(bytes_to_hex(b"ABC")) == b"ABC"


## 核心内容分步讲解

### Step 1: 理解CBC模式核心工作原理

CBC（Cipher Block Chaining，密码分组链接）是解决ECB模式安全性缺陷的经典分组密码模式，核心逻辑是**通过链式异或引入上下文关联**：
1. 初始化：生成一个随机的初始化向量（IV），长度与分组长度一致（如16字节）；
2. 加密流程：
   a. 第一个明文块$P_1$与IV异或：$P_1⊕IV$；
   b. 异或结果送入分组加密算法（如AES），得到第一个密文块$C_1=E_K(P_1⊕IV)$；
   c. 第二个明文块$P_2$与前一个密文块$C_1$异或：$P_2⊕C_1$，加密得到$C_2=E_K(P_2⊕C_1)$；
   d. 后续分组依次类推：$C_i=E_K(P_i⊕C_{i-1})$；
3. 解密流程：
   a. 第一个密文块$C_1$解密得到$D_K(C_1)$，与IV异或还原明文：$P_1=D_K(C_1)⊕IV$；
   b. 第二个密文块$C_2$解密得到$D_K(C_2)$，与$C_1$异或还原明文：$P_2=D_K(C_2)⊕C_1$；
   c. 后续分组：$P_i=D_K(C_i)⊕C_{i-1}$。
该模式通过链式结构让每个分组的加密结果依赖于前面所有分组，彻底解决ECB的模式泄露问题。


> 提示：可以类比为接力赛：每个分组的加密都需要前一个分组的“接力棒”（密文块），形成不可分割的整体

**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 2: 分析CBC模式的实现特性

基于CBC的链式工作原理，其实现层面有四个核心特性：
1. 必须使用IV：IV是CBC模式的核心，需满足两个要求——长度与分组一致、每次加密随机生成（不可重复使用相同IV+密钥）；
2. 加密串行化：由于每个分组的加密依赖前一个密文块，加密过程必须按顺序执行，无法并行处理；
3. 解密并行化：解密时每个分组仅依赖前一个密文块（已知），所有密文块可同时解密，效率较高；
4. 错误传播：单个密文块的错误会影响自身和下一个明文块的解密结果（后续分组不受影响），这一特性可用于检测数据篡改。
数学表达式上，CBC的加解密核心公式为：
加密：$C_i = E_K(P_i⊕C_{i-1}), C_0=IV$
解密：$P_i = D_K(C_i)⊕C_{i-1}$
其中$C_0$固定为IV，是链式加密的起始点。


> 提示：IV的随机性是CBC安全的关键：重复使用相同IV+密钥会导致第一个分组退化为ECB模式

**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 3: 掌握CBC模式的安全优势

相比ECB模式，CBC具备显著的安全优势，也是其成为主流模式的核心原因：
1. 语义安全性：相同明文在不同加密会话中会生成不同密文（因IV随机），攻击者无法通过密文模式推断明文内容；
2. 完整性检测：密文块的篡改会导致解密结果错误，可通过错误传播特性检测数据是否被篡改；
3. 抗重放攻击：IV的随机性使相同明文的密文不可复用，有效抵抗重放攻击；
4. 标准化支持：被TLS、IPSec、磁盘加密等主流安全协议广泛采用，是经过实践验证的安全模式。
CBC的唯一安全风险是IV的不当使用（如固定IV、可预测IV），只要保证IV随机且长度正确，CBC的安全性可满足大部分场景需求。


> 提示：CBC虽安全，但需配合消息认证码（MAC）使用，以抵抗padding oracle等针对性攻击

**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 4: 明确CBC模式的适用场景

CBC模式凭借良好的安全性和兼容性，成为应用最广泛的分组密码模式之一，主要适用场景包括：
1. 文件加密：本地文件、磁盘分区加密（如BitLocker支持CBC模式）；
2. 网络通信：TLS/SSL、VPN、IPSec等协议的加密层；
3. 数据库加密：数据库敏感字段、备份数据的加密；
4. 应用层加密：电商支付、金融交易、电子政务等核心业务数据加密；
5. 兼容性场景：需要兼容老旧系统的加密需求（CBC实现简单且兼容性好）。
核心适用条件：对加密并行性要求不高，对安全性和兼容性要求较高的场景。


> 提示：在高性能要求的场景（如大数据加密），可选择CTR/GCM等支持并行加密的模式替代CBC

**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 5: 实现CBC模式的加解密类

基于Python实现CBC模式的核心加解密逻辑，继承分组密码模式基类：
1. 初始化方法：指定分组长度（默认16字节），设置模式名称为"CBC"；
2. 加密方法encrypt：
   a. 处理IV：若未提供则随机生成，若提供则校验长度是否为分组长度；
   b. 对明文执行PKCS#7填充，确保长度为分组长度的整数倍；
   c. 初始化前一个块为IV，将IV拼接在密文开头（便于解密）；
   d. 遍历每个明文块：与前一个密文块异或→加密→拼接密文→更新前一个块；
3. 解密方法decrypt：
   a. 提取IV：若未提供则从密文开头截取，若提供则直接使用；
   b. 校验密文长度是否为分组长度的整数倍；
   c. 初始化前一个块为IV，遍历每个密文块：解密→与前一个块异或→拼接明文→更新前一个块；
   d. 执行PKCS#7去填充，得到原始明文。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
class CBCMode(BlockCipherMode):
    """密码分组链接模式 (Cipher Block Chaining Mode)"""
    
    def __init__(self, block_size: int = 16):
        super().__init__(block_size)
        self.name = "CBC"
    
    def encrypt(self, plaintext: bytes, key: bytes, iv: Optional[bytes] = None) -> bytes:
        cipher = SimpleAES(key)
        # 生成或使用提供的IV
        if iv is None:
            iv = self._generate_iv()
        elif len(iv) != self.block_size:
            raise ValueError(f"IV长度必须为{self.block_size}字节")
        # 填充明文
        padded_plaintext = self._pad_pkcs7(plaintext)
        # 分组加密
        ciphertext = iv  # 将IV放在密文前面
        blocks = self._split_blocks(padded_plaintext)
        previous_block = iv
        for block in blocks:
            # 与前一个密文分组异或
            xored_block = self._xor_blocks(block, previous_block)
            # 加密
            encrypted_block = cipher.encrypt_block(xored_block)
            ciphertext += encrypted_block
            previous_block = encrypted_block
        return ciphertext
    
    def decrypt(self, ciphertext: bytes, key: bytes, iv: Optional[bytes] = None) -> bytes:
        cipher = SimpleAES(key)
        if iv is None:
            # 从密文中提取IV
            if len(ciphertext) < self.block_size:
                raise ValueError("密文太短，无法包含IV")
            iv = ciphertext[:self.block_size]
            actual_ciphertext = ciphertext[self.block_size:]
        else:
            actual_ciphertext = ciphertext
        if len(actual_ciphertext) % self.block_size != 0:
            raise ValueError("密文长度必须是分组大小的倍数")
        # 分组解密
        plaintext = b''
        blocks = self._split_blocks(actual_ciphertext)
        previous_block = iv
        for block in blocks:
            # 解密
            decrypted_block = cipher.decrypt_block(block)
            # 与前一个密文分组异或
            xored_block = self._xor_blocks(decrypted_block, previous_block)
            plaintext += xored_block
            previous_block = block
        # 移除填充
        return self._unpad_pkcs7(plaintext)

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 6: 测试CBC模式的加解密正确性

编写测试代码验证CBC模式的加解密功能：
1. 准备测试数据：16字节密钥（符合AES-128要求）、53字节明文（非分组整数倍）；
2. 初始化CBC模式对象，默认分组长度16字节；
3. 执行加密操作：
   a. 模式自动生成随机IV，拼接在密文开头；
   b. 输出原文、原文长度、密文长度（IV+密文，53字节明文→16字节IV+64字节密文=80字节）；
   c. 输出密文前64位十六进制（便于查看）；
4. 执行解密操作：模式自动从密文开头提取IV，完成解密；
5. 验证结果：解密后的明文需与原始明文完全一致；
6. 额外验证：重复加密相同明文，两次密文应不同（因IV随机），验证语义安全性。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# 测试数据
key = b'0123456789abcdef'  # 16字节密钥
plaintext = b'Hello, this is a test message for block cipher modes!' 
print(f"原文: {plaintext}")
print(f"原文长度: {len(plaintext)} 字节")
print()
# 创建模式
mode = CBCMode()
# 加密
ciphertext = mode.encrypt(plaintext, key)
print(f"密文长度: {len(ciphertext)} 字节")
print(f"密文: {ciphertext.hex()[:64]}...")
# 解密
decrypted = mode.decrypt(ciphertext, key)
print(f"解密: {decrypted}")
# 验证
is_correct = plaintext == decrypted
print(f"正确性: {'✅' if is_correct else '❌'}")

# 预期输出：根据输入得到对应结果（请与讲解示例对照）

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 7: 总结CBC模式的使用规范

基于CBC模式的特性和安全要求，需遵守以下核心使用规范：
1. IV规范：
   a. 必须随机生成（使用加密安全的随机数生成器，如os.urandom）；
   b. 长度必须与分组长度一致（如AES-128对应16字节）；
   c. 不可重复使用相同IV+密钥（每次加密生成新IV）；
   d. IV无需保密，可随密文一起传输/存储；
2. 实现规范：
   a. 加密时将IV拼接在密文开头，便于解密提取；
   b. 解密前校验密文长度，确保包含完整IV和整数倍分组；
   c. 必须实现PKCS#7填充/去填充，处理非整数倍长度明文；
3. 安全增强：
   a. 配合MAC/HMAC使用，防止padding oracle攻击；
   b. 避免使用弱密钥，密钥需安全生成和存储；
   c. 加密大文件时分块处理，避免内存溢出。


> 提示：记住CBC的核心安全原则：随机IV + 正确填充 + 密钥安全 = 高安全性；固定IV = 退化为ECB = 严重安全风险

**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

## 综合示例：端到端流程 + 自测

In [ ]:
# 端到端模板：将主题核心操作封装成接口
# 你可以把 encrypt/decrypt 或 hash/sign/verify 等组合在一起

def pipeline(data: bytes) -> bytes:
    # TODO: 替换为你的端到端流程
    return hashlib.sha256(data).digest()

def check_pipeline():
    a = pipeline(b"hello")
    b = pipeline(b"hello")
    c = pipeline(b"hellp")
    assert a == b
    assert a != c
    print("端到端自测通过")  # 预期输出：端到端自测通过

check_pipeline()


## 小实验：敏感性（雪崩效应）

In [ ]:
def hamming_distance_bytes(a: bytes, b: bytes) -> int:
    if len(a) != len(b):
        raise ValueError("长度必须相同才能计算差异度")
    return sum(x != y for x, y in zip(a, b))

def flip_one_bit(data: bytes, bit_index: int = 0) -> bytes:
    if not data:
        return data
    byte_i = bit_index // 8
    bit_i = bit_index % 8
    byte_i = byte_i % len(data)
    mask = 1 << bit_i
    arr = bytearray(data)
    arr[byte_i] ^= mask
    return bytes(arr)

def core_transform(data: bytes) -> bytes:
    # TODO: 替换成你的核心变换
    # 示例：用 SHA-256 作为“对照组”
    return hashlib.sha256(data).digest()

base = b"hello world"
out1 = core_transform(base)
out2 = core_transform(flip_one_bit(base, 0))

print("输出长度(字节):", len(out1))  # 预期输出：32（若使用SHA-256）
print("差异度(字节):", hamming_distance_bytes(out1, out2))  # 预期输出：通常 > 10


## 附录A：最常用的数学与位运算背景

### 1. 模运算（Modular Arithmetic）

当我们写 $$a \equiv b \pmod n$$ 时，意思是 $a$ 与 $b$ 除以 $n$ 的余数相同，也就是 $n$ 整除 $a-b$。

常见等价写法：

- $a \bmod n$：把 $a$ 映射到 $0..n-1$ 的代表元
- 若 $a<0$，Python 仍保证 $a \bmod n \in [0,n-1]$

**一个很重要的直觉：** 模运算会“折叠”整数轴，把无限多的整数映射到有限集合，所以很多密码体制会用到它来构造“封闭”的运算空间。

### 2. 最大公因数与互质

若 $$\gcd(a,n)=1$$，我们称 $a$ 与 $n$ 互质（coprime）。互质非常重要，因为它意味着 $a$ 在模 $n$ 意义下存在乘法逆元 $a^{-1}$，满足：

$$a\cdot a^{-1} \equiv 1 \pmod n$$

### 3. 扩展欧几里得算法（Extended Euclid）

它不仅能算 $\gcd(a,b)$，还能找到整数 $x,y$ 使得：

$$ax+by=\gcd(a,b)$$

当 $\gcd(a,n)=1$ 时，$x$ 就是 $a$ 关于模 $n$ 的逆元（可能为负，需要再取模）。

### 4. 异或（XOR）

在字节层面，异或常写成 $c=a\oplus b$，它有几个极其重要的性质：

- $a\oplus a=0$
- $a\oplus 0=a$
- $(a\oplus b)\oplus b=a$（可逆性）

因此很多流密码/分组模式会用异或把“密钥流”与明文混合。

> 调试提示：如果你发现解密结果不等于明文，先检查“同一份密钥流是否被一致地使用”，以及字节对齐是否正确。


In [ ]:
# 附录A配套代码：gcd、逆元、异或操作的最小实现与自测

def egcd(a: int, b: int) -> Tuple[int, int, int]:
    """返回 (g, x, y) 使得 a*x + b*y = g = gcd(a,b)"""
    if b == 0:
        return (a, 1, 0)
    g, x1, y1 = egcd(b, a % b)
    x = y1
    y = x1 - (a // b) * y1
    return (g, x, y)

def modinv(a: int, n: int) -> int:
    g, x, _ = egcd(a, n)
    if g != 1:
        raise ValueError("不存在逆元：a 与 n 不互质")
    return x % n

print(math.gcd(3, 26))     # 预期输出：1
print(modinv(3, 26))       # 预期输出：9，因为 3*9=27≡1(mod26)

def xor_bytes(a: bytes, b: bytes) -> bytes:
    if len(a) != len(b):
        raise ValueError("xor 需要等长字节串")
    return bytes(x ^ y for x, y in zip(a, b))

p = b"ABC"
k = b"\x01\x01\x01"
c = xor_bytes(p, k)
p2 = xor_bytes(c, k)
print(c)   # 预期输出：b'@BA'（因为 0x41^1=0x40 等）
print(p2)  # 预期输出：b'ABC'


## 常见错误 / 踩坑点 / 调试提示

> 1. **编码问题**：`str`/`bytes` 混用导致哈希/加解密结果不一致。
>
> 2. **参数合法性**：例如需要互质、需要固定长度、需要随机数不可复用。
>
> 3. **端序与位序**：大端/小端混淆、位操作方向写反。
>
> 4. **测试不足**：缺少边界与异常路径测试。
>
> 5. **把演示当安全方案**：课程中的简化实现不等价于生产安全实现。

## 练习题（含更详细要求）

### 练习1：功能完善（必做）
- 目标：把你的核心函数做成“更好用”的版本  
- 要求：
  - 输入支持 `str` 与 `bytes`
  - 对非法参数给出清晰报错（`ValueError`）
  - 至少写 5 条 `assert`（覆盖正常/边界/异常）

### 练习2：最小攻防实验（推荐）
- 目标：实现一个最小的攻击/对抗脚本  
- 示例方向（任选其一）：
  - 暴力枚举 key space
  - 频率分析/统计偏差
  - 重放/篡改检测失败示例
  - 碰撞/相同摘要的构造（仅演示，不鼓励用于真实攻击）

### 练习3：写一段“工程化清单”（可选）
- 目标：把课程知识迁移到真实系统  
- 建议包含：
  - 参数选择（key 长度、随机数、模式）
  - 依赖库与实现来源（优先标准/权威实现）
  - 安全审计点（日志、异常、边界）


In [ ]:
# 练习参考答案框架（示例）
# 说明：这里只提供“结构”，你需要填入你自己的实现。

def solve_ex1(data):
    if isinstance(data, str):
        data_b = data.encode("utf-8")
    elif isinstance(data, (bytes, bytearray)):
        data_b = bytes(data)
    else:
        raise ValueError("只支持 str/bytes")
    return hashlib.sha256(data_b).hexdigest()

assert solve_ex1("a") == solve_ex1("a")
assert solve_ex1("a") != solve_ex1("b")
try:
    solve_ex1(123)
except ValueError:
    pass

print("练习1框架可运行")  # 预期输出：练习1框架可运行


## 小结

把今天的内容压缩成 3 句话：

1. 这个主题的核心变换是 $y=f(x,k)$（或 $$y=f(x,k)\bmod n$$）。
2. 正确实现需要重视：数据表示、参数合法性、测试向量与边界条件。
3. 真正理解来自：能写出最小攻防实验，并解释现象原因。


## 随堂小测（10题）
1. 这个主题的“密钥/参数”是什么？它决定了哪些行为？
2. 为什么需要模运算/置换/异或等操作？分别带来什么效果？
3. 在你的实现里，哪一步最容易写错？你如何用测试发现它？
4. 若攻击者拥有密文（或摘要），最直接的攻击方式是什么？
5. 在什么场景下“能解密”不等于“安全”？举一个例子。
6. 是否存在“重复使用随机数/nonce/IV”的风险？为什么危险？
7. 如果输入非常长，你的实现是否会变慢？瓶颈在哪？
8. 你能写出一个最小“对照实验”，让输出变化更直观吗？
9. 如果要用于真实系统，你会替换/增强哪一部分？
10. 用一句话总结：你学到的最重要概念是什么？

> 自评：能回答 7/10 且能跑通练习，就算达标。
